# Dedicated model for translation

## MADLAD

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer, BitsAndBytesConfig
import torch

In [ ]:
quantization_config = BitsAndBytesConfig(
    # load_in_8bit = True
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",            # Better accuracy than standard fp4
    # bnb_4bit_use_double_quant=True        # Compresses the quantization constants
)

In [ ]:
model_name = 'google/madlad400-10b-mt'
model = T5ForConditionalGeneration.from_pretrained(
    model_name, 
    device_map="cuda", 
    torch_dtype=torch.float16,
    quantization_config=quantization_config
)
tokenizer = T5Tokenizer.from_pretrained(model_name)

In [ ]:
text = "<2id> What a successful toast, it looks so delicious!"
input_ids = tokenizer(text, return_tensors="pt").input_ids.to('cuda')
outputs = model.generate(input_ids=input_ids)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

## NLBB

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
import torch

/root/bukan-skripsi/codes/test-translate/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# quantization_config = BitsAndBytesConfig(
#     load_in_8bit = True
#     # load_in_4bit=True,
#     # bnb_4bit_compute_dtype=torch.float16,
#     # bnb_4bit_quant_type="nf4",            # Better accuracy than standard fp4
#     # bnb_4bit_use_double_quant=True        # Compresses the quantization constants
# )

In [2]:
# Load tokenizer and model directly
model_name = "facebook/nllb-200-3.3B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name, 
    # quantization_config=quantization_config, 
    dtype=torch.float16,
    device_map="auto"
    )

W0816 14:50:34.418000 1193143 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0816 14:50:34.455000 1193143 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 1016/1016 [01:41<00:00,  9.98it/s]


In [ ]:
# Prepare text and set language tokens (e.g., English to French)
# text = "What a successful toast, it looks so delicious!"
tokenizer.src_lang = "eng_Latn"
inputs = tokenizer(text, return_tensors="pt").to("cuda")

# Generate translation using target language code token (e.g., french: fra_Latn)
translated_tokens = model.generate(
    **inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids("ind_Latn")
)
output = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
print(output)

Cuaca yang indah, membuatku ingin memeluk diriku sendiri
